@author: chaoping

# Final project: Virginia state senate analysis
All data retrieved April 2025: <br>
[Population data](https://redistrictingdatahub.org/dataset/virginia-block-pl-94171-2020-by-table/): based on the decennial census at the Census Block level on 2020 Census Redistricting Data

[2020 County data](https://redistrictingdatahub.org/dataset/virginia-county-pl-94171-2020/): from 2020 Census Redistricting Data (P.L. 94-171) Shapefiles

[State Senate District data](https://redistrictingdatahub.org/dataset/2021-senate-of-virginia-districts-approved-plan/): 2021 State Senate Approved Plan

[2020 election data](https://redistrictingdatahub.org/dataset/vest-2020-virginia-precinct-boundaries-and-election-results-shapefile/)**:**  VEST 2020 Virginia precinct and election results

[2018 election data](https://redistrictingdatahub.org/dataset/vest-2018-virginia-precinct-and-election-results/)**:**  VEST 20218 Virginia precinct and election results


In [1]:
import pandas as pd
import geopandas as gpd
import maup
from maup import smart_repair
import time
import os

In [11]:
import warnings
warnings.filterwarnings('ignore')

maup.progress.enabled = True

## Loading shape files and explore the data

In [2]:
# Load population file
population_df = gpd.read_file("./va_pl2020_b/va_pl2020_p2_b.shp")

#Load voting age population file
vap_df = gpd.read_file("./va_pl2020_b/va_pl2020_p4_b.shp")

#Load 2020 election data
vest20_df = gpd.read_file("./va_vest_20/va_vest_20.shp")

#Load county data
county_df = gpd.read_file("./va_pl2020_cnty/va_pl2020_cnty.shp")

#Load state senate district
sen_df = gpd.read_file("./va_sldu_adopted_2021/SCV FINAL SD.shp")

Check column names

In [3]:
print(population_df.columns)
print(vap_df.columns)
print(vest20_df.columns)
print(county_df.columns)
print(sen_df.columns)

Index(['GEOID20', 'SUMLEV', 'LOGRECNO', 'GEOID', 'COUNTY', 'P0020001',
       'P0020002', 'P0020003', 'P0020004', 'P0020005', 'P0020006', 'P0020007',
       'P0020008', 'P0020009', 'P0020010', 'P0020011', 'P0020012', 'P0020013',
       'P0020014', 'P0020015', 'P0020016', 'P0020017', 'P0020018', 'P0020019',
       'P0020020', 'P0020021', 'P0020022', 'P0020023', 'P0020024', 'P0020025',
       'P0020026', 'P0020027', 'P0020028', 'P0020029', 'P0020030', 'P0020031',
       'P0020032', 'P0020033', 'P0020034', 'P0020035', 'P0020036', 'P0020037',
       'P0020038', 'P0020039', 'P0020040', 'P0020041', 'P0020042', 'P0020043',
       'P0020044', 'P0020045', 'P0020046', 'P0020047', 'P0020048', 'P0020049',
       'P0020050', 'P0020051', 'P0020052', 'P0020053', 'P0020054', 'P0020055',
       'P0020056', 'P0020057', 'P0020058', 'P0020059', 'P0020060', 'P0020061',
       'P0020062', 'P0020063', 'P0020064', 'P0020065', 'P0020066', 'P0020067',
       'P0020068', 'P0020069', 'P0020070', 'P0020071', 'P002

## Clean the data

First, we put everything into utm crs.

In [4]:
population_df = population_df.to_crs(population_df.estimate_utm_crs())
vap_df = vap_df.to_crs(vap_df.estimate_utm_crs())
county_df = county_df.to_crs(county_df.estimate_utm_crs())
sen_df = sen_df.to_crs(sen_df.estimate_utm_crs())
vest20_df = vest20_df.to_crs(vest20_df.estimate_utm_crs())

Check with maup.doctor()

In [5]:
maup.doctor(population_df)

True

In [6]:
maup.doctor(vap_df)

True

In [7]:
maup.doctor(county_df)

True

In [8]:
maup.doctor(sen_df)

True

In [9]:
maup.doctor(vest20_df)

There are 81 overlaps.
There are 549 holes.


False

We need to do smart repair with our 2020 election data.

We first nest within counties.

In [12]:
final_df = smart_repair(vest20_df, nest_within_regions = county_df)

100%|████████████████████████████████████████| 133/133 [00:00<00:00, 225.84it/s]


Snapping all geometries to a grid with precision 10^( -5 ) to avoid GEOS errors.


100%|████████████████████████████████████████| 133/133 [00:00<00:00, 542.93it/s]


Identifying overlaps...


100%|█████████████████████████████████████| 2786/2786 [00:02<00:00, 1251.54it/s]


Resolving overlaps and filling gaps...


100%|████████████████████████████████████████| 133/133 [00:00<00:00, 573.22it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 2: 100%|███████████████| 2/2 [00:00<00:00,  2.54it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 3: 100%|███████████████| 1/1 [00:00<00:00, 67.63it/s]


Found a component of the region at index 3 that does not intersect any geometry assigned to that region.


Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 4: 100%|██████████████| 1/1 [00:00<00:00, 122.93it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 7: 100%|██████████████| 1/1 [00:00<00:00, 150.24it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 9: 100%|███████████████| 1/1 [00:00<00:00, 60.04it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 10: 100%|█████████████| 3/3 [00:00<00:00, 103.79it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 13: 100%|█████████████| 1/1 [00:00<00:00, 138.58it/s]
Gaps to fill: 0it [00:00, ?it/s]
Gaps to simplify in region 14: 100%|███████████

In [13]:
final_df = smart_repair(final_df, min_rook_length = 30)

Snapping all geometries to a grid with precision 10^( -5 ) to avoid GEOS errors.
Identifying overlaps...


100%|█████████████████████████████████████| 2494/2494 [00:00<00:00, 3735.39it/s]


Resolving overlaps...
Filling gaps...


Gaps to simplify: 100%|███████████████████████████| 1/1 [00:00<00:00,  7.50it/s]
Gaps to fill: 0it [00:00, ?it/s]

Converting small rook adjacencies to queen...



100%|███████████████████████████████████████████| 4/4 [00:00<00:00, 3029.47it/s]


In [14]:
maup.doctor(final_df)

100%|██████████████████████████████████████| 2477/2477 [00:03<00:00, 776.24it/s]


True